# 💬 Chatbot con Cache-Augmented Generation (CAG)

### Diplomado de Inteligencia Artificial — Módulo 3 (NLP y Deep Learning)
**Universidad de Santiago de Chile (USACH)**

Material adaptado por **Diego Fischer**.

---

En este cuaderno construimos un chatbot usando **Cache-Augmented Generation (CAG)**.

La idea es sencilla y elegante. En vez de recuperar fragmentos de un documento en cada consulta —como haría **RAG** (*Retrieval-Augmented Generation*)—, cargamos el documento **una sola vez** dentro del modelo y guardamos su representación interna (las claves y valores de atención, o *KV cache*). Después, cada pregunta del usuario se responde **continuando** a partir de esa caché ya calculada, sin volver a procesar todo el documento.

Esto sirve cuando el conocimiento cabe en la ventana de contexto del modelo: se gana velocidad y se evita el recuperador externo, a cambio de quedar limitados al documento cacheado.

Usaremos:
- **Mistral** (`mistralai/Mistral-7B-Instruct-v0.1`) como modelo de lenguaje.
- El mecanismo de caché dinámica (`DynamicCache`) que entrega `transformers`.

> **Requisito:** una cuenta de Hugging Face con un *token* de acceso y una GPU con memoria suficiente (Mistral-7B en FP16 ocupa ~15 GB; una T4 queda muy justa, una L4 o A100 va holgada).

# Fase 1: Dependencias y función de generación

Importamos las librerías y definimos una función de generación *token a token*. En cada paso alimentamos el último token junto con la caché de claves/valores (`past_key_values`), tomamos el token más probable (decodificación *greedy*) y repetimos hasta alcanzar el largo máximo o el token de fin de secuencia.

In [ ]:
import torch
import copy
from transformers import AutoTokenizer, AutoModelForCausalLM

def generate(model, input_ids: torch.Tensor, past_key_values,
             max_new_tokens: int = 200) -> torch.Tensor:
    """Genera texto token a token reutilizando una caché KV ya construida.
    Devuelve solo los tokens nuevos (sin el prompt de entrada)."""
    device = model.device
    origin_len = input_ids.shape[-1]
    input_ids = input_ids.to(device)
    output_ids = input_ids.clone()
    next_token = input_ids

    # eos_token_id puede ser un entero o una lista (p. ej. en Llama-3);
    # lo normalizamos a lista para cortar la generación correctamente.
    eos_ids = model.config.eos_token_id
    if isinstance(eos_ids, int):
        eos_ids = [eos_ids]

    with torch.no_grad():
        for _ in range(max_new_tokens):
            out = model(input_ids=next_token,
                        past_key_values=past_key_values,
                        use_cache=True)
            logits = out.logits[:, -1, :]
            # Decodificación greedy. Para muestreo, reemplazar por:
            #   probs = torch.softmax(logits / temperature, dim=-1)
            #   token = torch.multinomial(probs, num_samples=1)
            token = torch.argmax(logits, dim=-1, keepdim=True)
            output_ids = torch.cat([output_ids, token], dim=-1)
            past_key_values = out.past_key_values
            next_token = token.to(device)

            if eos_ids is not None and token.item() in eos_ids:
                break

    return output_ids[:, origin_len:]

# Fase 2: Construir la caché de conocimiento

Aquí está el corazón de CAG. La función `get_kv_cache` hace un *prefill*: pasa el prompt por el modelo una vez, con `use_cache=True`, y devuelve la caché de claves/valores resultante junto con el largo del prompt en tokens.

En las versiones actuales de `transformers`, el modelo crea internamente un `DynamicCache` y lo entrega en `out.past_key_values`, así que no necesitamos construirlo a mano (esto además nos evita los cambios de firma que ha tenido esa clase entre versiones).

In [ ]:
def get_kv_cache(model, tokenizer, prompt: str):
    """Ejecuta un prefill del modelo con el prompt dado y devuelve:
    - past_key_values: la caché KV (DynamicCache) del prompt.
    - origin_len: largo del prompt en tokens.
    """
    device = model.device
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    input_ids = inputs["input_ids"]
    origin_len = input_ids.shape[-1]

    with torch.no_grad():
        out = model(input_ids=input_ids, use_cache=True)

    return out.past_key_values, origin_len

# Fase 3: Cargar el modelo (Mistral)

Cargamos `mistralai/Mistral-7B-Instruct-v0.1` en FP16 si hay GPU disponible. El parámetro `device_map="auto"` deja que la librería `accelerate` ubique el modelo, por lo que **no** llamamos a `model.to(...)` manualmente.

In [ ]:
from google.colab import userdata

model_name = "mistralai/Mistral-7B-Instruct-v0.1"
HF_TOKEN = userdata.get("HF_TOKEN")

tokenizer = AutoTokenizer.from_pretrained(model_name, token=HF_TOKEN)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto",
    token=HF_TOKEN,
)
model.eval()
print(f"Modelo cargado: {model_name} (device: {model.device})")

# Fase 4: Preparar el documento y cachearlo

Definimos el documento que hará de base de conocimiento (aquí, un extracto de un artículo sobre clasificación de tráfico de red con CNN) y armamos el *prompt de conocimiento* con el formato de instrucción de Mistral (`[INST] ... [/INST]`).

Un detalle importante: dejamos el prompt **abierto**, terminando en `Pregunta: `. Ese prefijo estático —instrucciones + documento— es lo que cacheamos una sola vez. Cada pregunta se agregará después, justo antes de generar la respuesta.

In [ ]:
doc_text = """
ABSTRACT Network traffic classification plays a crucial role in detecting malware threats. However,
most existing research focuses on extracting statistical features from the network traffic, ignoring the rich
information contained within raw packet capture (pcap) files. To achieve higher accuracy in malware traffic
classification, a novel approach is proposed that fully utilizes the information contained in the pcap files by
representing them with images and then training deep Convolutional Neural Networks (CNN) to learn the
features automatically and classify them with higher accuracy. Selected fields of the IP headers in network
sessions are transformed into 50 × 50 RGB images. These images serve as input to CNN, and malware
samples are grouped by class or malware name. The model is initially trained and validated on the MCFP
dataset with more than 140 malware classes and subsequently tested on separate datasets, namely USTCTFC2016,
Taltech.ee MedBIoT, and IEEE-Mirai. The macro F1 scores and accuracy of this method are
significantly higher than the baseline statistical-feature based approach both in the validation dataset and in
the test datasets from different sources. The results of this research have the potential to be extended beyond
malware classification to enable the classification of various types of network traffic data.
INDEX TERMS Malware classification, network traffic classification, deep learning, convolutional neural
network
D. CNN ARCHITECTURE
VGG [31] was initially used to classify images from the
ImageNet [32] dataset. The dataset contains more than
FIGURE 4. Image distribution bottom 50 of malware classes of MCFP
dataset.
14 million images and 1,000 classes. Packet images are
simpler than those of ImageNet. The number of classes
142 is also less than ImageNet. Therefore, it should be
less challenging to classify compared to ImageNet. The
decision was made to simplify VGG to meet the project’s
requirements.
The following Figure 5 shows the architecture of our CNN
model. The input to the network was 50 × 50 pixel RGB
images. This was followed by a sequence of convolutional
blocks consisting of convolutional layers interspersed with
max pooling layers. The convolutional filter sizes were 3 × 3
and the number of filters in the convolutional layers were
64, 64, 128, and 128 respectively. To reduce overfitting,
dropout layers with a rate of 0.5 were added after every
convolutional block. The convolutional features were finally
fed into a flatten layer and two fully connected dense
layers with 4096 units each before an output layer with
142 channels for 142 malware classes. Figure 6 is the
summary of the Deep Convolutional Neural Network which
is more detailed description of Figure 5. There are over
93 trainable parameters in this neural network.
FIGURE 5. Deep convolutional neural network architecture diagram.
E. TRAINING CNN MODEL
A random 80-20 split was performed on the MCFP dataset,
resulting in 1,242,468 images in the training set and 310,618
images in the validation dataset. AdamW optimizer in
Pytorch library was used with 1.0 × 10−5 as the learning
rate and 0.01 as the weight decay factor. Cross-Entropy was
used as the loss function. The batch size was 128. The CNN
models were implemented in Pytorch and trained on 4 Nvidia
58034 VOLUME 12, 2024
R. E. Davis Jr. et al.: Classifying Malware Traffic Using Images and Deep CNN
FIGURE 6. Detailed summary of CNN model.
GPUs. The model converged fairly quickly within 2 epochs.
To show the gradual progress of learning, instead of every
epoch, we sampled loss and accuracy for every 100 batches
of training, which is equivalent to every 12,800 samples.
A random 100 batches in the validation dataset is selected
when computing the validation loss and validation accuracy.
The following Figures 7 and 8 show the progress of training
and validation loss and accuracy on every 100 batches as a
unit.Validation loss and validation accuracy is computed after
every 100 batches of training in the beginning. This explains
why validation result is better than training in the beginning.
However, as the loss function and accuracy converge, the
difference between the two is very small. It is observed that
there is no sign of overfitting. Both training and validation
loss stayed stable in the later stage of training. A more
detailed result of training and validation will be shown in the
result section of this paper.
FIGURE 7. Training and validation accuracy history.
F. BASELINE METHOD
The performance of the image-based CNN model was chosen
for comparison against a commonly used method in the
FIGURE 8. Training and validation loss history.
literature, where statistical features are extracted from the
packet captures and machine learning algorithms are applied
to this structured dataset. Netml [33] is a Python library that
extracts 10 of the most common statistics in the literature:
flow duration, number of packets per second, number of
bytes per second, and the following statistics of packet sizes:
mean, standard deviation, the first to the third quantiles, the
minimum and the maximum. A Python script written with
netml library to extract these statistical features from the
same MCFP dataset. Please, note that the first 25 megabytes
was used in large PCAP files for generating images. We did
the same when extracting the statistical features. Another
thing to note is that these statistical features are flow-based
and images are generated based on sessions. Therefore, the
statistically based dataset has 4,472,196 samples, which is
about twice the number of samples as the images. Random
Forest, Support Vector Machine, and XGBoost are applied on
this statistically based dataset. A detailed result will be shown
in the following result section.
"""

instrucciones = (
    "Eres un experto en ciberseguridad que analiza el uso de imágenes para detectar "
    "vulnerabilidades en la transmisión de datos por internet. Responde en español, "
    "de forma clara y precisa, destacando los detalles relevantes. Usa únicamente la "
    "información del documento; si algo no aparece en él, indícalo."
)

# Prefijo estático (instrucciones + documento). El tokenizer agrega el token <s>
# de inicio automáticamente, por eso no lo escribimos a mano.
system_prompt = f"[INST] {instrucciones}\n\nDocumento:\n{doc_text}\n\nPregunta: "

# Cacheamos el prefijo una sola vez
contexto_cache, origin_len = get_kv_cache(model, tokenizer, system_prompt)
print(f"Caché de conocimiento construida ({origin_len} tokens).")

# Fase 5: Responder preguntas

Para cada pregunta partimos de una **copia limpia** de la caché del documento (`copy.deepcopy`), agregamos los tokens de la pregunta y el cierre `[/INST]`, y generamos la respuesta. Al copiar la caché en cada consulta, las preguntas no se contaminan entre sí.

Nota: la pregunta se tokeniza con `add_special_tokens=False` para no insertar un segundo token de inicio en medio del prompt.

In [ ]:
def answer_question(question: str, max_new_tokens: int = 200) -> str:
    # Copia limpia de la caché del documento para esta consulta
    local_cache = copy.deepcopy(contexto_cache)

    # Tokenizamos solo la pregunta + el cierre de instrucción de Mistral
    resto = f"{question} [/INST]"
    input_ids = tokenizer.encode(resto, return_tensors="pt",
                                 add_special_tokens=False).to(model.device)

    gen_ids = generate(model, input_ids, local_cache, max_new_tokens=max_new_tokens)
    return tokenizer.decode(gen_ids[0], skip_special_tokens=True).strip()

In [ ]:
question1 = "Explícame qué clases de amenazas se pueden identificar usando imágenes."
answer1 = answer_question(question1)
print("Pregunta:", question1)
print("Respuesta:", answer1)

In [ ]:
question2 = "¿Qué arquitectura de red neuronal se usó y cómo se transformaron los datos?"
answer2 = answer_question(question2)
print("Pregunta:", question2)
print("Respuesta:", answer2)

# Discusión y uso responsable

Algunas ideas para cerrar y discutir en clase:

- **CAG vs. RAG.** CAG brilla cuando el conocimiento es acotado y estable (cabe en el contexto). Si la base crece o cambia seguido, RAG con recuperación sigue siendo mejor. Se pueden combinar.
- **Límites del modelo.** El chatbot solo "sabe" lo que está en el documento cacheado. Aun así puede **alucinar** o mezclar conocimiento previo del modelo; sus respuestas no deben tomarse como verdad establecida, sobre todo en un dominio sensible como la ciberseguridad.
- **Decodificación.** Usamos *greedy* por simplicidad y reproducibilidad. Activar muestreo (temperatura) da respuestas más variadas pero menos deterministas.
- **Costo de memoria.** La caché crece con el largo del documento; documentos muy grandes pueden no caber junto con el modelo en la GPU.

---

### Licencia

Este material se distribuye bajo licencia **Creative Commons Atribución 4.0 Internacional (CC BY 4.0)**.  
Puede compartirse y adaptarse, incluso con fines comerciales, siempre que se reconozca la autoría de forma adecuada.

Material elaborado por **Diego Fischer** para la Universidad de Santiago de Chile (USACH).  
Más información: https://creativecommons.org/licenses/by/4.0/deed.es